In [1]:
# 08_advanced_experiments.ipynb - Full ready-to-paste cell(s)
# Purpose: Aggregate results from previous notebooks (distillation, calibration, explainability, rule_mapping)
# and produce advanced analyses: ECE, reliability diagrams, per-class metrics, model size/latency, explainability correlations.
# Paste this entire block into the notebook and run.

import os, json, time, math, glob
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, brier_score_loss

# ---------------- CONFIG ----------------
ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2")
DISTILL_ROOT = ROOT / "outputs" / "distillation"
CALIB_ROOT = ROOT / "outputs" / "calibration"
EXPLAIN_ROOT = ROOT / "outputs" / "explainability"
RULE_ROOT = ROOT / "outputs" / "rule_mapping"
ADV_OUT = ROOT / "outputs" / "advanced_experiments"
STUDENT_MODEL_ID = "distil_distilbert-base-multilingual-cased"
SAMPLE_INFERENCE_BATCH = 8  # for latency estimate
N_LATENCY_REPS = 30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ensure_dir = lambda p: p.mkdir(parents=True, exist_ok=True) or p

# ---------------- utilities ----------------
def find_prediction_file(exp_dir: Path):
    """Try several candidate filenames used across pipeline to find predictions csv."""
    candidates = [
        "predictions.csv",
        "student_test_file_predictions_topk_k3_calibrated.csv",
        "student_test_file_predictions_topk_k3.csv",
        "student_test_file_predictions.csv",
        "predictions_test.csv",
        "student_predictions.csv",
        "predictions.csv"
    ]
    for c in candidates:
        p = exp_dir / c
        if p.exists(): return p
    # fallback: any csv with 'prediction' or 'pred' in name
    for p in exp_dir.glob("*.csv"):
        if "pred" in p.name.lower() or "prediction" in p.name.lower():
            return p
    return None

def normalize_pred_df(pred_df: pd.DataFrame, label_map: dict=None, inv_label_map: dict=None):
    """Normalize many prediction formats to consistent fields:
       file_path, pred_label (str), pred_label_id (int), probs (np.array)
       If probs not present, tries to infer from columns named like prob_*, or from a 'pred_probs' JSON-like string.
    """
    df = pred_df.copy()
    if 'file_path' not in df.columns:
        # try filename-like columns
        possible = [c for c in df.columns if 'file' in c.lower() or 'path' in c.lower()]
        if possible:
            df = df.rename(columns={possible[0]:'file_path'})
        else:
            raise ValueError("predictions file has no file_path column")
    # find prob columns
    prob_cols = [c for c in df.columns if c.lower().startswith("prob") or c.lower().startswith("p_") or c.lower().startswith("proba")]
    probs_list = []
    pred_label_ids = []
    pred_label_strs = []
    for idx,row in df.iterrows():
        probs = None
        if 'pred_probs' in df.columns and pd.notnull(row['pred_probs']):
            try:
                val = row['pred_probs']
                if isinstance(val, str):
                    probs = np.array(json.loads(val))
                elif isinstance(val, (list, tuple, np.ndarray)):
                    probs = np.array(val)
            except Exception:
                pass
        if probs is None and prob_cols:
            arr=[]
            for c in prob_cols:
                v = row.get(c, 0.0)
                arr.append(float(v) if pd.notnull(v) else 0.0)
            probs = np.array(arr)
        # fallback: if pred_label_id exists, one-hot
        pli = None
        if 'pred_label_id' in df.columns and not pd.isnull(row.get('pred_label_id')):
            try: pli = int(row['pred_label_id'])
            except Exception: pli = None
        if probs is None:
            num_labels = len(label_map) if label_map else (np.max(df['pred_label_id'].fillna(-1).astype(int).values)+1 if 'pred_label_id' in df.columns else None)
            if pli is not None and num_labels is not None:
                p = np.zeros((num_labels,), dtype=float); p[pli]=1.0; probs = p
            else:
                # uniform fallback
                probs = np.ones((max(2, num_labels or 2),), dtype=float)
                probs = probs / probs.sum()
        # ensure numpy
        if not isinstance(probs, np.ndarray):
            probs = np.array(probs)
        probs_list.append(probs)
        # pred labels
        if 'pred_label' in df.columns and pd.notnull(row.get('pred_label')):
            pred_label_strs.append(str(row.get('pred_label')))
            if pli is None and label_map:
                pred_label_ids.append(int(label_map.get(str(row.get('pred_label')), -1)))
            else:
                pred_label_ids.append(int(pli) if pli is not None else int(np.argmax(probs)))
        else:
            pred_label_ids.append(int(pli) if pli is not None else int(np.argmax(probs)))
            pred_label_strs.append(inv_label_map[pred_label_ids[-1]] if inv_label_map is not None and pred_label_ids[-1] in inv_label_map else str(pred_label_ids[-1]))
    df['pred_probs_norm'] = probs_list
    df['pred_label_id_norm'] = pred_label_ids
    df['pred_label_norm'] = pred_label_strs
    return df

def expected_calibration_error(probs: np.ndarray, labels: np.ndarray, n_bins=15):
    """Compute top-label ECE (standard). probs: (n_samples, n_classes). labels: (n_samples,) ints."""
    confidences = probs.max(axis=1)
    preds = probs.argmax(axis=1)
    accuracies = (preds == labels).astype(float)
    bins = np.linspace(0.0,1.0,n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        mask = (confidences>bins[i]) & (confidences<=bins[i+1])
        if mask.sum()==0: continue
        bin_acc = accuracies[mask].mean()
        bin_conf = confidences[mask].mean()
        ece += (mask.sum()/len(probs)) * abs(bin_conf - bin_acc)
    return float(ece)

def classwise_ece(probs: np.ndarray, labels: np.ndarray, n_bins=10):
    """Compute per-class ECE by treating each class as a one-vs-rest calibration."""
    K = probs.shape[1]
    eces = {}
    for k in range(K):
        confidences = probs[:,k]
        truek = (labels==k).astype(float)
        bins = np.linspace(0.0,1.0,n_bins+1)
        ece = 0.0
        for i in range(n_bins):
            mask = (confidences>bins[i]) & (confidences<=bins[i+1])
            if mask.sum()==0: continue
            bin_acc = truek[mask].mean()
            bin_conf = confidences[mask].mean()
            ece += (mask.sum()/len(probs)) * abs(bin_conf - bin_acc)
        eces[k] = float(ece)
    return eces

def plot_reliability_diagram(probs, labels, outpath: Path, n_bins=15, title=None):
    """Reliability diagram (top-label)."""
    ensure_dir(outpath.parent)
    confidences = probs.max(axis=1)
    preds = probs.argmax(axis=1)
    accuracies = (preds==labels).astype(float)
    bins = np.linspace(0.0,1.0,n_bins+1)
    bin_centers=[]
    bin_accs=[]
    bin_confs=[]
    for i in range(n_bins):
        mask = (confidences>bins[i]) & (confidences<=bins[i+1])
        if mask.sum()==0:
            bin_centers.append((bins[i]+bins[i+1])/2)
            bin_accs.append(np.nan)
            bin_confs.append(np.nan)
        else:
            bin_centers.append((bins[i]+bins[i+1])/2)
            bin_accs.append(accuracies[mask].mean())
            bin_confs.append(confidences[mask].mean())
    plt.figure(figsize=(6,6))
    plt.plot([0,1],[0,1],"k:", label="perfect")
    plt.plot(bin_centers, [0 if math.isnan(x) else x for x in bin_accs], marker="o", label="accuracy")
    plt.bar(bin_centers, [0 if math.isnan(c) else (c - (0 if math.isnan(a) else a)) for c,a in zip(bin_confs, bin_accs)], alpha=0.4, width=1.0/n_bins, label="confidence - accuracy")
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    if title: plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(outpath), dpi=150)
    plt.close()

# ---------------- gather experiments ----------------
ensure_dir(ADV_OUT)
languages = sorted([p.name for p in (ROOT/"data_splits").iterdir() if p.is_dir()])
print("Languages discovered:", languages)

experiments_index = {}
for lang in languages:
    exp_folder = DISTILL_ROOT / lang / STUDENT_MODEL_ID
    if not exp_folder.exists():
        print(f"[WARN] missing distillation folder for {lang} at {exp_folder}; skipping language")
        continue
    pred_file = find_prediction_file(exp_folder)
    calib_folder = CALIB_ROOT / lang / STUDENT_MODEL_ID
    explain_folder = EXPLAIN_ROOT / lang / STUDENT_MODEL_ID
    rule_folder = RULE_ROOT / lang / STUDENT_MODEL_ID
    experiments_index[lang] = {
        "distill_folder": str(exp_folder),
        "predictions_csv": str(pred_file) if pred_file else None,
        "calibration_folder": str(calib_folder) if calib_folder.exists() else None,
        "explain_folder": str(explain_folder) if explain_folder.exists() else None,
        "rule_folder": str(rule_folder) if rule_folder.exists() else None
    }

# Save index
json.dump(experiments_index, open(ADV_OUT/"experiments_index.json","w",encoding="utf8"), indent=2)
print("Saved experiments_index.json to", ADV_OUT/"experiments_index.json")

# ---------------- per-language analysis ----------------
summary_rows = []
for lang, info in experiments_index.items():
    print("\n--- ANALYZING", lang, "---")
    lang_out = ADV_OUT / lang / STUDENT_MODEL_ID
    ensure_dir(lang_out)

    # load label_map
    label_map_path = ROOT / "data_splits" / lang / "label_map.json"
    if not label_map_path.exists():
        print(f"[WARN] label_map missing for {lang}")
        continue
    lm = json.load(open(label_map_path, encoding="utf8"))
    label_map = {str(k):int(v) for k,v in lm.get("label_map",{}).items()}
    inv_label_map = {int(v):str(k) for k,v in lm.get("label_map",{}).items()}
    labels_list = [inv_label_map[i] for i in range(len(inv_label_map))]

    # load predictions
    pred_csv = info.get("predictions_csv")
    if not pred_csv:
        print(f"[WARN] predictions CSV not found for {lang} -> skipping")
        continue
    df_pred_raw = pd.read_csv(pred_csv)
    try:
        df_pred = normalize_pred_df(df_pred_raw, label_map=label_map, inv_label_map=inv_label_map)
    except Exception as e:
        print(f"[ERROR] normalizing predictions for {lang}: {e}")
        continue

    # Try to load the test split to get gold labels if present
    test_csv = ROOT / "data_splits" / lang / "test.csv"
    df_test = pd.read_csv(test_csv) if test_csv.exists() else None

    # align preds to test rows by file_path; if test missing fallback to pred_df only
    if df_test is not None:
        # merge on file_path with basename fallback
        merged = pd.merge(df_test, df_pred, on="file_path", how="left", suffixes=("_gold","_pred"))
        # for missing merged rows (NaNs), try basename match
        missing_mask = merged['pred_label_id_norm'].isnull()
        if missing_mask.any():
            for i,row in merged[missing_mask].iterrows():
                base = Path(row['file_path']).name
                cand = df_pred[df_pred['file_path'].str.endswith(base)]
                if len(cand)>0:
                    merged.at[i, 'pred_label_id_norm'] = cand.iloc[0]['pred_label_id_norm']
                    merged.at[i, 'pred_label_norm'] = cand.iloc[0]['pred_label_norm']
                    merged.at[i, 'pred_probs_norm'] = cand.iloc[0]['pred_probs_norm']
        df_all = merged
    else:
        # create dummy gold column if not present
        df_pred['gold_label'] = None
        df_pred['gold_label_id'] = None
        df_all = df_pred

    # Build arrays for evaluation where gold exists
    mask_has_gold = df_all['label'].notnull() if 'label' in df_all.columns else df_all['gold_label'].notnull() if 'gold_label' in df_all.columns else pd.Series([False]*len(df_all))
    if mask_has_gold.sum() == 0:
        print(f"[WARN] No gold labels present for {lang}; we will still report stats on preds.")
    # prepare y_true, y_pred, probs
    y_true = []
    y_pred = []
    probs = []
    file_paths = []
    for idx, row in df_all.iterrows():
        file_paths.append(row.get('file_path') or row.get('file_path_pred') or None)

        val = row.get('pred_probs_norm', None)
        if val is None:
            p = np.ones((len(label_map),), dtype=float) / len(label_map)
        else:
            try:
                arr = np.array(val, dtype=float)
                if arr.ndim == 0:  # scalar → make uniform
                    p = np.ones((len(label_map),), dtype=float) / len(label_map)
                else:
                    p = arr
            except Exception:
                p = np.ones((len(label_map),), dtype=float) / len(label_map)
        probs.append(p)

        if 'label' in row and not pd.isnull(row['label']):
            # label string -> id
            lab = row['label']
            if lab in label_map: y_true.append(int(label_map[lab]))
            else:
                try: y_true.append(int(row.get('label_id', 0)))
                except Exception: y_true.append(0)
        elif 'gold_label' in row and not pd.isnull(row['gold_label']):
            lab = row['gold_label']
            if lab in label_map: y_true.append(int(label_map[lab]))
            else:
                try: y_true.append(int(row.get('gold_label_id', 0)))
                except Exception: y_true.append(0)
        else:
            y_true.append(None)
        # prediction
        try:
            y_pred.append(int(row['pred_label_id_norm']))
        except Exception:
            try: y_pred.append(int(np.argmax(probs[-1])))
            except Exception: y_pred.append(0)
    probs = np.vstack([p if isinstance(p, np.ndarray) else np.array(p) for p in probs])
    # compute top-label preds if required
    preds_from_probs = probs.argmax(axis=1)

    # evaluation where gold present
    valid_idx = [i for i,y in enumerate(y_true) if y is not None]
    y_true_arr = np.array([y_true[i] for i in valid_idx]) if len(valid_idx)>0 else np.array([])
    y_pred_arr = np.array([y_pred[i] for i in valid_idx]) if len(valid_idx)>0 else np.array([])
    probs_arr = probs[valid_idx] if len(valid_idx)>0 else np.array([])

    # metrics
    if len(valid_idx)>0:
        acc = float(accuracy_score(y_true_arr, y_pred_arr))
        p,r,f,_ = precision_recall_fscore_support(y_true_arr, y_pred_arr, labels=list(range(len(label_map))), zero_division=0)
        per_class = {inv_label_map[i]: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f[i])} for i in range(len(inv_label_map))}
        cm = confusion_matrix(y_true_arr, y_pred_arr, labels=list(range(len(label_map)))).tolist()
        # calibration metrics
        ece = expected_calibration_error(probs_arr, y_true_arr, n_bins=15)
        class_ece = classwise_ece(probs_arr, y_true_arr, n_bins=10)
        brier = float(np.mean([brier_score_loss((y_true_arr==k).astype(int), probs_arr[:,k]) for k in range(probs_arr.shape[1])]))
    else:
        acc = None; per_class={}; cm=None; ece=None; class_ece={}; brier=None

    # Save per-experiment summary
    summary = {
        "language": lang,
        "n_examples_total": len(df_all),
        "n_with_gold": len(valid_idx),
        "accuracy": acc,
        "per_class": per_class,
        "confusion_matrix": cm,
        "ece_top_label": ece,
        "classwise_ece": class_ece,
        "brier_score_mean": brier
    }
    json.dump(summary, open(lang_out/"experiment_summary.json","w",encoding="utf8"), indent=2)

    # Save CSV of per-file preds + gold + probs
    df_save = pd.DataFrame({
        "file_path": file_paths,
        "gold_label_id": [y_true[i] for i in range(len(y_true))],
        "pred_label_id": [int(y_pred[i]) for i in range(len(y_pred))],
        "pred_label_from_probs": [int(preds_from_probs[i]) for i in range(len(preds_from_probs))],
        "pred_probs": [p.tolist() for p in probs]
    })
    df_save.to_csv(lang_out/"predictions_merged.csv", index=False, encoding="utf8")

    # Reliability diagram
    if ece is not None and len(valid_idx)>0:
        plot_reliability_diagram(probs_arr, y_true_arr, lang_out/"reliability_diagram.png", n_bins=15, title=f"{lang}: reliability diagram (ECE={ece:.4f})")

    # confusion matrix plot
    if cm is not None:
        cm_arr = np.array(cm)
        plt.figure(figsize=(6,5))
        sns.heatmap(cm_arr, annot=True, fmt="d", xticklabels=labels_list, yticklabels=labels_list, cmap="Blues")
        plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"{lang} Confusion Matrix")
        plt.tight_layout()
        plt.savefig(lang_out/"confusion_matrix.png", dpi=150)
        plt.close()

    # Latency estimate: load model (if present) and run small samples (be defensive)
    model_dir = DISTILL_ROOT / lang / STUDENT_MODEL_ID / "best_model"
    latency_ms = None
    model_size_mb = None
    if model_dir.exists():
        # compute size on disk
        total_bytes = sum(f.stat().st_size for f in model_dir.rglob("*") if f.is_file())
        model_size_mb = total_bytes / (1024*1024)
        # try to do a small latency test (use tokenizer + model)
        try:
            from transformers import AutoTokenizer, AutoModelForSequenceClassification
            tok = AutoTokenizer.from_pretrained(str(model_dir), use_fast=True)
            m = AutoModelForSequenceClassification.from_pretrained(str(model_dir)).to(DEVICE).eval()
            # prepare a small batch from some random texts from df_all (or fallback short dummy)
            sample_texts = []
            if 'file_path' in df_all.columns:
                for x in df_all['file_path'].dropna().tolist()[:SAMPLE_INFERENCE_BATCH]:
                    try:
                        txt = Path(x).read_text(encoding="utf8", errors="ignore")
                        sample_texts.append(txt[:200])
                    except Exception:
                        sample_texts.append("Hello world")
            if len(sample_texts)==0:
                sample_texts = ["Hello world"]*SAMPLE_INFERENCE_BATCH
            # tokenize batch
            enc = tok(sample_texts, padding=True, truncation=True, max_length=256, return_tensors='pt')
            input_ids = enc['input_ids'].to(DEVICE); att = enc['attention_mask'].to(DEVICE)
            # warmup
            with torch.no_grad():
                for _ in range(3): _ = m(input_ids=input_ids, attention_mask=att)
            # timed runs
            t0=time.time()
            for _ in range(N_LATENCY_REPS):
                with torch.no_grad():
                    _ = m(input_ids=input_ids, attention_mask=att)
            t1=time.time()
            elapsed = (t1-t0)
            latency_ms = (elapsed / (N_LATENCY_REPS * input_ids.size(0))) * 1000.0
            # free model
            del m; del tok; torch.cuda.empty_cache()
        except Exception as e:
            print("Latency test failed for", lang, ":", e)
            latency_ms = None

    # Merge explainability diagnostics if available
    expl_path = info.get("explain_folder")
    explain_correlation = {}
    if expl_path:
        expl_metrics = Path(expl_path) / "explainability_metrics_per_file.csv"
        if expl_metrics.exists():
            df_expl = pd.read_csv(expl_metrics)
            # join by file_path and compute correlation between deletion_auc and correctness
            # build correctness vector aligned with df_expl (for files present in df_save)
            merged_expl = pd.merge(df_save, df_expl, on="file_path", how="inner")
            if not merged_expl.empty and 'deletion_auc' in merged_expl.columns:
                merged_expl['correct'] = (merged_expl['gold_label_id'] == merged_expl['pred_label_id']).astype(int)
                # correlation
                corr = merged_expl[['deletion_auc','correct']].corr().iloc[0,1]
                explain_correlation['deletion_auc_vs_correct'] = float(corr) if not np.isnan(corr) else None
                # group stats
                grouped = merged_expl.groupby('correct')['deletion_auc'].describe().to_dict()
                explain_correlation['deletion_auc_by_correct'] = grouped

    # Save per-lang summary row
    summary_rows.append({
        "language": lang,
        "n_examples_total": len(df_all),
        "n_with_gold": len(valid_idx),
        "accuracy": acc,
        "ece_top_label": ece,
        "brier": brier,
        "model_size_mb": model_size_mb,
        "latency_ms_per_sample": latency_ms,
        "explainability_corr": explain_correlation
    })

    print(f"[{lang}] acc={acc} ece={ece} model_size_mb={model_size_mb} latency_ms={latency_ms}")

# write aggregated summary CSV/JSON
pd.DataFrame(summary_rows).to_csv(ADV_OUT/"advanced_experiments_summary.csv", index=False)
json.dump(summary_rows, open(ADV_OUT/"advanced_experiments_summary.json","w",encoding="utf8"), indent=2)
print("Saved advanced_experiments summary to", ADV_OUT)

# Done
print("\n[ALL DONE] Advanced experiments completed. Check:", ADV_OUT)


Languages discovered: ['English', 'Hindi', 'Marathi']
Saved experiments_index.json to /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/advanced_experiments/experiments_index.json

--- ANALYZING English ---
[English] acc=0.5813953488372093 ece=0.41860465116279066 model_size_mb=519.986759185791 latency_ms=0.6238202253977458

--- ANALYZING Hindi ---
[Hindi] acc=0.8064516129032258 ece=0.19354838709677424 model_size_mb=519.9802112579346 latency_ms=0.9241918722788492

--- ANALYZING Marathi ---
[Marathi] acc=0.5625 ece=0.4375 model_size_mb=519.9768447875977 latency_ms=0.907475749651591
Saved advanced_experiments summary to /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/advanced_experiments

[ALL DONE] Advanced experiments completed. Check: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/advanced_experiments
